In [1]:
import numpy as np
import pandas as pd
import os
import re
from collections import Counter, defaultdict
import networkx as nx
import folium
from folium.plugins import MarkerCluster
import geopy
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import spacy
from spacy.lang.fr.stop_words import STOP_WORDS
import string
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())
import plotly.graph_objects as go
import plotly.express as px
import csv

# Download the stopwords
nltk.download('stopwords')

# Path to the data folder
# ROOT="C:/Users/lolon/OneDrive/EPFL/RO-1ST YEAR 2023-2024/SHS/HUM-450_part_2/HUM-450"
ROOT="/Users/selmabenhassine/Desktop/MA2/SHS/HUM-450"
path_data=ROOT+"/data/data_Impresso"

# Year, party, query word to change
year = "1966-1980"
party = "PSS"
query = ["parti", "socialiste", "suisse"]

# Path to results folder
results_data = ROOT + f"/data/data_Language/Sentiments"


# File name
#file_name = "1890_1910_PartiSocialiste_PartiSocialisteSuisse_PSS"
#file_name = "1890_1910_Radicaux_PartiRadicalDemocratique_GaucheRadicale_GR_PRD"
#file_name = "1890_1910_UnionConservatrice_PartiPopulaireCatholique"

#file_name = "1945_1960_PartiDesPaysansArtisansEtIndependants_PAI_PartiDesPaysansArtisansBourgeois_PAB"
#file_name = "1945_1965_PartiConservateurPopulaire_PCP"
#file_name = "1945_1965_PartiRadical-Democratique_PRD"
#file_name = "1945_1965_PartiSocialiste_PSS"

#file_name = "1966_1980_PAB_PAI_UDC_UnionDemocratiqueDuCentre"
#file_name = "1966_1980_PDC_PartiDemocrateCHretien_PCCS_PartiConservateurChretienSocial"
#file_name = "1966_1980_PRD_PartiRadicalDemocratique_GR_radicaux"
#file_name = "1966_1980_PSS_PartiSocialiste"

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/selmabenhassine/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
def load_database(path):
    """
    Simple method for loading the database
    """
    database = pd.read_csv(path, sep=';')
    database['persons_mentioned']=database['persons_mentioned'].apply(lambda x: x if pd.isnull(x) else x[1:-1].replace("'", '').replace(", ",",").split(','))
    database['locations_mentioned']=database['locations_mentioned'].apply(lambda x: x if pd.isnull(x) else x[1:-1].replace("'", '').replace(", ",",").split(','))
    return database




In [3]:
#Load the data
extraction=load_database(path_data+f"/{year}/{file_name}.csv")
extraction.head()
#print(extraction.shape)
print(extraction.date[0][:4])



1966


In [4]:

# Define a regular expression pattern for French letters
french_letters_pattern = re.compile(r'^[a-zA-ZÀ-ÿ]+$')

# French stopwords provided by NLTK
french_stopwords = set(stopwords.words('french'))

# Deselect some stop words
deselect_stop_words = ['n\'', 'ne','pas','plus','personne','aucun','ni','aucune','rien']
for w in deselect_stop_words:
    if w in french_stopwords:
        french_stopwords.remove(w)
    else:
        continue

#Extract the token
N_token=10
extratced_string_before=[]
extratced_string_after=[]
relevant_years=[]

for i in range(0, extraction.shape[0]):
    #Prepare the data
    token=str.lower(extraction.content[i])
    token=re.sub(r'[();,"|%\'\'$*&:«».#^/<>+-]', ' ', token)
    token=re.split(r'\s+', token)
    white_ind=[]
    for j in range(0,len(token)):
        if (token[j]!=''):
            white_ind.append(int(j))
    token=np.take(token, white_ind, 0)

    # Filter out words that do not contain only French letters
    token = [word for word in token if french_letters_pattern.match(word)] 

    # Split each token further into individual words
    sentence=[]
    token = [word for sublist in token for word in sublist.split()]
    sentence.append(token)

    #Findex the indexes
    indexes=[]
    for j in range(len(sentence[0])):
        # Check if the current index is already within N_token distance of another index
        if not any(abs(j - existing_index) < N_token for existing_index in indexes):
            # Ensure we check within bounds
            if (sentence[0][j] in query and
                ( (j-2 >= 0 and sentence[0][j-2] in query) or
                  (j-1 >= 0 and sentence[0][j-1] in query) or
                  (j+1 < len(sentence[0]) and sentence[0][j+1] in query) or
                  (j+2 < len(sentence[0]) and sentence[0][j+2] in query) )):
                # Define the minisentence bounds
                start = max(0, j-2)
                end = min(len(sentence[0]), j+3)  # +3 because end index is exclusive
                
                # Extract the minisentence
                minisentence = sentence[0][start:end]
                
                # Check for duplicates within the minisentence
                query_words_in_minisentence = [word for word in minisentence if word in query]
                if len(query_words_in_minisentence) == len(set(query_words_in_minisentence)):
                    indexes.append(j)

    if not(len(indexes)==0):
        for ind in indexes:
            start_index = ind
            # Extract words before the query index
            before_query = []
            j = start_index
            sentence_length = N_token
            while len(before_query) < sentence_length+1:
                if j >= 0:
                    word = sentence[0][j]
                    before_query.insert(0, word)    
                else:
                    before_query.insert(0, '')
                j -= 1
            
            # Extract words after the query index
            after_query = []
            j = start_index + 1
            sentence_length = N_token
            while len(after_query) < sentence_length:
                if j < len(sentence[0]):
                    word = sentence[0][j]
                    after_query.append(word)     
                else:
                    after_query.append('')
                j += 1

            extratced_string_before.append(before_query)
            extratced_string_after.append(after_query)
            relevant_years.append(int(extraction.date[i][:4]))

In [5]:
print(len(extratced_string_before))
print(len(extratced_string_after))
print(len(relevant_years))

year

372
372
372


'1966-1980'

In [6]:
# Find the length of the longest row
max_length = max(len(row) for row in extratced_string_before)

# Pad each row to match the length of the longest row
for row in extratced_string_before:
    while len(row) < max_length:
        row.append('')  # You can use any padding value you prefer

# Find the length of the longest row
max_length = max(len(row) for row in extratced_string_after)

# Pad each row to match the length of the longest row
for row in extratced_string_after:
    while len(row) < max_length:
        row.append('')  # You can use any padding value you prefer        


extratced_string_before=np.asmatrix(extratced_string_before)
extratced_string_after=np.asmatrix(extratced_string_after)
token_matrix=np.concatenate([extratced_string_before,extratced_string_after], axis=1)

# Assuming token_matrix is a numpy array of matrices
token_matrix_strings = [' '.join(filter(lambda x: x.strip(), np.array(row).flatten())) for row in token_matrix]

# Print the first row as an example
print(token_matrix_strings)
print(len(token_matrix_strings))
print(relevant_years)


['place exceptionnelle son journal pouvait être simplement la voix du parti socialiste ou bien cédant au réflexe professoral et suisse romand', 'voient obligés d en tenir compte et d abord le parti socialiste non sans quelque irritation parfois chez les vieux routiers', 'présence des conseillers fédéraux spûhler et tschudi a lausanne le parti socialiste suisse a tenu un congrès à ouvert m pierre', 'd une aile droite ou d une aile gauche du parti socialiste suisse déclarait samedi avec prudence le conseiller national fritz', 'fondé de l initiative mais il en dissocie pratiquement le parti socialiste suisse en demandant que les sections cantonales gardent leur', 'départ immédiat de m paul chaudet mais cela dit le parti socialiste suisse sachant qu il a à l égal des', 'les socialistes suisses et les finances fédérales la correspondance du parti socialiste suisse communique présidé par le conseiller national fritz gruetter', 'par le conseiller national fritz gruetter le comité directeur du

In [7]:
data_output = [['Sentence', 'Year']]
for i in range(len(token_matrix_strings)):
    data_output.append([token_matrix_strings[i], relevant_years[i]])
data_output

[['Sentence', 'Year'],
 ['place exceptionnelle son journal pouvait être simplement la voix du parti socialiste ou bien cédant au réflexe professoral et suisse romand',
  1966],
 ['voient obligés d en tenir compte et d abord le parti socialiste non sans quelque irritation parfois chez les vieux routiers',
  1966],
 ['présence des conseillers fédéraux spûhler et tschudi a lausanne le parti socialiste suisse a tenu un congrès à ouvert m pierre',
  1966],
 ['d une aile droite ou d une aile gauche du parti socialiste suisse déclarait samedi avec prudence le conseiller national fritz',
  1966],
 ['fondé de l initiative mais il en dissocie pratiquement le parti socialiste suisse en demandant que les sections cantonales gardent leur',
  1966],
 ['départ immédiat de m paul chaudet mais cela dit le parti socialiste suisse sachant qu il a à l égal des',
  1966],
 ['les socialistes suisses et les finances fédérales la correspondance du parti socialiste suisse communique présidé par le conseiller n

In [8]:
save_path = results_data + f'/sentiment_{year}_{party}.csv'
with open(save_path, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(data_output)

#sentiment_strings = token_matrix_strings